Step 1: Import required libraries

In [2]:
%pip install mysql-connector-python
import os
os.environ['MYSQL_PASSWORD'] = 'naya123'

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import mysql.connector
import os
from pathlib import Path

Step 2: Establish MySQL database connection

In [4]:
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password=os.environ['MYSQL_PASSWORD'],      # Set MYSQL_PASSWORD in your environment
    database='db_churn'      # Target database
)

Step 3: Load raw CSV data into DataFrame


In [5]:
df = pd.read_csv(Path.cwd().parent / 'data' / 'Customer_Churn_Data.csv')

print(f"Total records loaded: {len(df)}")
print(df.head())

Total records loaded: 6418
  Customer_ID  Gender  Age Married        State  Number_of_Referrals  \
0   19877-DEL    Male   35      No        Delhi                    7   
1   58353-MAH  Female   45     Yes  Maharashtra                   14   
2   25063-WES    Male   51      No  West Bengal                    4   
3   59787-KAR    Male   79      No    Karnataka                    3   
4   28544-TAM  Female   80      No   Tamil Nadu                    3   

   Tenure_in_Months Value_Deal Phone_Service Multiple_Lines  ...  \
0                27        NaN           Yes             No  ...   
1                13        NaN           Yes            Yes  ...   
2                35     Deal 5           Yes             No  ...   
3                21     Deal 4           Yes             No  ...   
4                 8        NaN           Yes             No  ...   

    Payment_Method Monthly_Charge Total_Charges Total_Refunds  \
0      Credit Card           65.6        593.30          0.00   
1

Step 4: Handle missing/null values before insertion

In [6]:
df = df.where(pd.notnull(df), None)

Step 5: Insert data row by row into staging table

In [7]:
cursor = conn.cursor()
cursor.execute("TRUNCATE TABLE stg_churn")

for _, row in df.iterrows():
    cursor.execute("""
        INSERT INTO stg_churn VALUES 
        (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,
         %s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, tuple(row))

Step 6: Commit transaction and close connection

In [8]:
conn.commit()
cursor.close()
conn.close()

print("Data successfully imported into stg_churn table!")

Data successfully imported into stg_churn table!
